# 🎓 Student Performance Prediction

Predicting students' final grade (G3) using the UCI Student Performance Dataset.

**Dataset:** [Kaggle - Student Performance Data](https://www.kaggle.com/datasets/devansodariya/student-performance-data)

**Target:** G3 (Final Grade, 0–20)

**Features dropped:**
- `G1`, `G2` — intermediate period grades (data leakage).
- `school` — school identifier, not a generalisable predictor.

**Pipeline:**
1. Load & clean data
2. EDA + Correlation heatmap
3. Preprocessing (encode categoricals, scale)
4. Train & compare 8 algorithms
5. Auto-select best model (lowest RMSE)
6. Overfitting analysis (CV gap + learning curve)
7. Final predictions on test set

## 1. Install & Import Libraries

> Run this cell first. All other cells also re-import what they need locally, so they can be re-run in isolation without restarting the kernel.

In [ ]:
# !pip install scikit-learn pandas matplotlib seaborn

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, cross_val_score,
                                     learning_curve, KFold)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

print('✅ All libraries imported successfully')

## 2. Load & Clean Dataset

In [ ]:
import os, pandas as pd

def load_student_data(path='student_data.csv'):
    if not os.path.exists(path):
        url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/student-mat.csv'
        print(f'File not found locally, trying URL: {url}')
        raw = pd.read_csv(url, sep=None, engine='python')
    else:
        with open(path, 'r') as f:
            first_line = f.readline()
        sep = ';' if ';' in first_line else ','
        print(f'Detected separator: {repr(sep)}')
        raw = pd.read_csv(path, sep=sep)
    if raw.shape[1] == 1:
        raise ValueError('Only 1 column loaded — wrong separator.')
    return raw

df_raw = load_student_data('student_data.csv')
print(f'✅ Loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')

COLS_TO_DROP = ['G1', 'G2', 'school']
df = df_raw.drop(columns=COLS_TO_DROP)
print(f'Features after drop: {df.shape[1] - 1} features + 1 target (G3)')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
import numpy as np
print('Shape:', df.shape)
print('\nMissing values:\n', df.isnull().sum())
print('\nG3 stats:\n', df['G3'].describe())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].hist(df['G3'], bins=21, color='steelblue', edgecolor='white')
axes[0,0].set_title('G3 Distribution'); axes[0,0].set_xlabel('Grade')
axes[0,1].scatter(df['absences'], df['G3'], alpha=0.4, color='steelblue')
axes[0,1].set_title('Absences vs G3')
axes[1,0].boxplot([df[df['studytime']==st]['G3'].values for st in sorted(df['studytime'].unique())],
                  labels=sorted(df['studytime'].unique()))
axes[1,0].set_title('Study Time vs G3')
axes[1,1].boxplot([df[df['failures']==f]['G3'].values for f in sorted(df['failures'].unique())],
                  labels=sorted(df['failures'].unique()))
axes[1,1].set_title('Past Failures vs G3')
plt.tight_layout(); plt.savefig('eda_plots.png', dpi=150); plt.show()

## 4. Correlation Heatmap

In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns

numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(14, 11))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap (G1/G2 removed)')
plt.tight_layout(); plt.savefig('correlation_heatmap.png', dpi=150); plt.show()

## 5. Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df_model = df.copy()
le = LabelEncoder()
cat_cols = df_model.select_dtypes(include=['object']).columns.tolist()
print('Encoding:', cat_cols)
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

df_model = df_model[df_model['G3'] > 0].reset_index(drop=True)
print(f'Rows after removing G3=0: {len(df_model)}')

X = df_model.drop(columns=['G3'])
y = df_model['G3']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'Train: {X_train_sc.shape}  Test: {X_test_sc.shape}')

## 6. Model Comparison — 8 Algorithms

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

MODELS = {
    'Linear Regression' : LinearRegression(),
    'Ridge'             : Ridge(),
    'Lasso'             : Lasso(),
    'Decision Tree'     : DecisionTreeRegressor(random_state=42),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=100, random_state=42),
    'SVR'               : SVR(),
    'KNN'               : KNeighborsRegressor(),
}

results = {}
print(f"{'Model':25s} | {'RMSE':>6} | {'MAE':>6} | {'R²':>6}")
print('-' * 55)
for name, model in MODELS.items():
    model.fit(X_train_sc, y_train)
    preds = model.predict(X_test_sc)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    r2   = r2_score(y_test, preds)
    results[name] = {'model': model, 'RMSE': rmse, 'MAE': mae, 'R2': r2}
    print(f'{name:25s} | {rmse:6.3f} | {mae:6.3f} | {r2:6.3f}')

best_name  = min(results, key=lambda k: results[k]['RMSE'])
best_model = results[best_name]['model']
print(f'\n🏆 Best model: {best_name}  '      f'(RMSE={results[best_name]["RMSE"]:.3f}, R²={results[best_name]["R2"]:.3f})')

## 7. Overfitting Analysis

Two complementary techniques:
1. **CV gap** — compare mean 5-fold CV RMSE vs hold-out test RMSE. Gap > 0.5 ⇒ possible overfitting.
2. **Learning curve** — plot train vs validation RMSE as training size grows.

In [ ]:
# Local imports so this cell works even if Cell 1 was not run
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print(f"{'Model':25s} | {'CV RMSE (train)':>16} | {'Test RMSE':>10} | Gap")
print('-' * 70)

for name, info in results.items():
    m = info['model']
    cv_scores = cross_val_score(m, X_train_sc, y_train,
                               cv=kf,
                               scoring='neg_root_mean_squared_error')
    cv_rmse   = -cv_scores.mean()
    test_rmse = info['RMSE']
    gap  = test_rmse - cv_rmse
    flag = '⚠️  overfit?' if gap > 0.5 else '✅'
    print(f'{name:25s} | {cv_rmse:16.3f} | {test_rmse:10.3f} | {gap:+.3f}  {flag}')

In [ ]:
# Local imports so this cell works even if Cell 1 was not run
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    best_model,
    X_train_sc, y_train,
    cv=5,
    scoring='neg_root_mean_squared_error',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)

train_rmse = -train_scores.mean(axis=1)
val_rmse   = -val_scores.mean(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_rmse, 'o-', color='royalblue', label='Train RMSE')
plt.plot(train_sizes, val_rmse,   's-', color='tomato',    label='Validation RMSE')
plt.fill_between(train_sizes,
                 -train_scores.min(axis=1), -train_scores.max(axis=1),
                 alpha=0.1, color='royalblue')
plt.fill_between(train_sizes,
                 -val_scores.min(axis=1), -val_scores.max(axis=1),
                 alpha=0.1, color='tomato')
plt.xlabel('Training set size')
plt.ylabel('RMSE')
plt.title(f'Learning Curve — {best_name}')
plt.legend()
plt.tight_layout()
plt.savefig('learning_curve.png', dpi=150)
plt.show()

final_gap = val_rmse[-1] - train_rmse[-1]
print(f'Train RMSE @ full data : {train_rmse[-1]:.3f}')
print(f'Val   RMSE @ full data : {val_rmse[-1]:.3f}')
print(f'Gap                    : {final_gap:+.3f}')
if final_gap > 0.5:
    print('⚠️  Noticeable gap — model may be overfitting.')
else:
    print('✅ Train/Val gap is acceptable — no significant overfitting detected.')

## 8. Final Predictions on Test Set

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

best_model.fit(X_train_sc, y_train)
y_pred = best_model.predict(X_test_sc)
y_pred_rounded = np.clip(np.round(y_pred).astype(int), 0, 20)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
print(f'=== {best_name} — Final Test Results ===')
print(f'  RMSE : {rmse:.3f}')
print(f'  MAE  : {mae:.3f}')
print(f'  R²   : {r2:.3f}')

preds_df = pd.DataFrame({
    'Actual G3'   : y_test.values,
    'Predicted G3': y_pred_rounded,
    'Error'       : (y_pred_rounded - y_test.values)
}).reset_index(drop=True)
print('\nFirst 20 predictions:')
print(preds_df.head(20).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, y_pred, alpha=0.5, color='steelblue', edgecolors='white')
mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual G3'); axes[0].set_ylabel('Predicted G3')
axes[0].set_title(f'Actual vs Predicted — {best_name}'); axes[0].legend()

residuals = y_test.values - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, color='tomato', edgecolors='white')
axes[1].axhline(0, color='black', lw=1.5, linestyle='--')
axes[1].set_xlabel('Predicted G3')
axes[1].set_ylabel('Residual (Actual − Predicted)')
axes[1].set_title('Residual Plot')

plt.tight_layout(); plt.savefig('prediction_results.png', dpi=150); plt.show()

## 9. Feature Importance (if supported)

In [ ]:
import matplotlib.pyplot as plt, pandas as pd

if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X.columns)
    importances = importances.sort_values(ascending=False)
    plt.figure(figsize=(10, 6))
    importances.plot(kind='bar', color='steelblue', edgecolor='white')
    plt.title(f'Feature Importances — {best_name}')
    plt.ylabel('Importance')
    plt.tight_layout(); plt.savefig('feature_importances.png', dpi=150); plt.show()
    print(importances.to_string())
else:
    print(f'{best_name} does not expose feature_importances_. Skipping.')